In [1]:
import os
import sys
from openai import OpenAI
from pathlib import Path
import json
import asyncio


project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src.rag_settings import Settings, RunSummary

# This moves up one level from the notebook's location to find the root folder
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.rag_pipeline import (
    chunk_documents, chunk_text, build_index, embed_batch,
    retrieve, ask_rag,
    cost_usd, DEFAULT_SYSTEM,
    calc_golden_hit_cost_lat, dense_search,
    create_collection, upsert_collection, retrive_from_collection, fetch_all_chunks_from_qdrant, 
    simple_tokenize, build_bm25_index, bm25_search,
    rrf_fuse,  
)


Connected to Qdrant at https://91ecc16f-269a-4046-b44c-5a190057...


In [ ]:
# Global values setup
# 1. Define the folder path
folder_path = Path("../docs/Dale_Carnigale")
coll_name = "Book_DALE_CARN"

assert os.environ.get("OPENAI_API_KEY") # "Set OPENAI_API_KEY before running this notebook"
#os.environ["OPENAI_API_URL"] = "https://openai.vocareum.com/v1"
#os.environ["OPENAI_API_KEY"] = "voc-204283627021403739729016a89b188c0a8a4.37364158"

_client = OpenAI()

_rag_settings = Settings()
_rag_settings.GENERATE_ANSWER_RAG = False

SYSTEM = (
    "You are a helpful assistant. Answer the user's question using the "
    "provided context. Cite the source id in square brackets after any fact you use."
)

k = 5

## Code for BM25 data generation using Chunks stored in Qdrant

In [3]:
# Get all chunks from Qdrant/ vector DB
all_chunks = []
all_chunks = fetch_all_chunks_from_qdrant(coll_name)

print(f"Length of all chunks: {len(all_chunks)}")

# Store the data in output folder for BM25 geenration
import pickle

bm25_tokenized_corpus = build_bm25_index(all_chunks)
with open(folder_path/"output/bm25_index.pkl", "wb") as f:
    pickle.dump(bm25_tokenized_corpus, f)

Length of all chunks: 348


In [4]:
# 2. Load instantly in future runs
with open(folder_path/"output/bm25_index.pkl", "rb") as f:
    bm25_corpus = pickle.load(f)
    
print("📊 Corpus Size (Total Docs):", bm25_corpus.corpus_size)
print("📏 Individual Doc Lengths:", bm25_corpus.doc_len)
print("📐 Average Doc Length (avgdl):", bm25_corpus.avgdl)

📊 Corpus Size (Total Docs): 348
📏 Individual Doc Lengths: [58, 55, 50, 51, 55, 58, 58, 58, 59, 56, 50, 56, 62, 50, 55, 54, 59, 48, 52, 47, 54, 58, 60, 48, 53, 53, 58, 54, 51, 56, 55, 52, 51, 50, 52, 52, 58, 52, 52, 51, 56, 52, 52, 47, 55, 57, 54, 54, 61, 62, 61, 61, 56, 60, 56, 64, 64, 61, 66, 64, 58, 24, 57, 51, 55, 54, 56, 49, 55, 55, 49, 53, 46, 54, 52, 50, 50, 53, 18, 50, 52, 53, 48, 52, 53, 48, 49, 52, 47, 50, 10, 58, 57, 51, 65, 60, 57, 62, 59, 55, 53, 54, 57, 52, 52, 56, 59, 60, 56, 58, 61, 62, 65, 65, 55, 60, 61, 59, 59, 55, 57, 57, 52, 58, 62, 54, 59, 60, 65, 59, 56, 62, 47, 57, 57, 55, 51, 59, 65, 57, 55, 53, 59, 60, 33, 57, 61, 56, 58, 59, 61, 53, 59, 59, 55, 62, 56, 48, 55, 57, 53, 51, 56, 52, 31, 23, 18, 49, 54, 45, 51, 48, 47, 52, 49, 44, 53, 51, 48, 51, 50, 41, 22, 50, 53, 56, 54, 51, 55, 57, 54, 51, 53, 48, 47, 54, 56, 61, 53, 51, 55, 58, 58, 51, 47, 50, 45, 56, 62, 54, 47, 41, 45, 49, 59, 51, 54, 56, 62, 66, 71, 57, 60, 60, 59, 57, 56, 58, 59, 56, 53, 57, 58, 56, 59, 5

In [5]:
QUERY01 = "Who were the publishers of the book?" 
QUERY01 = "Which university did psychologist Professor James V. McConnell belong to?"
QUERY02 =  "when did the Course originate?" 
qry = []
qry.append(QUERY01)
#qry.append(QUERY02)
q_vec = []
q_vec = embed_batch(qry)
#q_vec

In [6]:
dense_results = await dense_search(QUERY01, coll_name, k)

In [7]:
dense_results

[{'chunk_id': 'HtoWF_04_Part02_02.txt#9',
  'source_id': 'HtoWF_04_Part02_02.txt',
  'text': 'ile that comes from within, the kind of smile that will bring a good price in the marketplace.\n\nProfessor James V. McConnell, a psychologist at the University of Michigan, expressed his feelings about a smile. "People who smile," he said, "tend to manage, teach and sell more effectively, and to rais',
  'score': 0.3120165},
 {'chunk_id': 'HtoWF_04_Part02_02.txt#14',
  'source_id': 'HtoWF_04_Part02_02.txt',
  'text': 'as about to be graduated from Purdue University. After several phone conversations I learned that he had several offers from other companies, many of them larger and better known than mine. I was delighted when he accepted my offer. After he started on the job, I asked him why he had chosen us over ',
  'score': 0.28579056},
 {'chunk_id': 'HtoWF_02WhythisBook.txt#14',
  'source_id': 'HtoWF_02WhythisBook.txt',
  'text': 'dults in even one college in the land, it has escaped my at

In [8]:
bm25_result = await bm25_search(QUERY01, bm25_corpus, all_chunks, k)

In [9]:
bm25_result

[{'chunk_id': 'HtoWF_04_Part02_02.txt#9',
  'source_id': 'HtoWF_04_Part02_02.txt',
  'score': 29.173718288516284,
  'text': 'ile that comes from within, the kind of smile that will bring a good price in the marketplace.\n\nProfessor James V. McConnell, a psychologist at the University of Michigan, expressed his feelings about a smile. "People who smile," he said, "tend to manage, teach and sell more effectively, and to rais'},
 {'chunk_id': 'HtoWF_02WhythisBook.txt#32',
  'source_id': 'HtoWF_02WhythisBook.txt',
  'score': 19.407848423238146,
  'text': 'urces. Stating the thing broadly, the human individual thus lives far within his limits. He possesses powers of various sorts which he habitually fails to use."\n\n— Professor William James (Harvard University)\n\nThose powers which you "habitually fail to use"—the sole purpose of this book is to help y'},
 {'chunk_id': 'HtoWF_04_Part02_02.txt#30',
  'source_id': 'HtoWF_04_Part02_02.txt',
  'score': 12.625230805588709,
  'text': 'yoursel

In [10]:
print(f"Query: {QUERY01}\n\n====================================\n")
# List comprehension to get structured dictionaries
rrf_score = rrf_fuse([dense_results, bm25_result], k=60, top_n=k)
print(f"Reranked Fused score:")
print(rrf_score)


Query: Which university did psychologist Professor James V. McConnell belong to?


Reranked Fused score:
[{'chunk_id': 'HtoWF_04_Part02_02.txt#9', 'source_id': 'HtoWF_04_Part02_02.txt', 'text': 'ile that comes from within, the kind of smile that will bring a good price in the marketplace.\n\nProfessor James V. McConnell, a psychologist at the University of Michigan, expressed his feelings about a smile. "People who smile," he said, "tend to manage, teach and sell more effectively, and to rais', 'score': 0.3120165, 'rrf_score': 0.03278688524590164}, {'chunk_id': 'HtoWF_04_Part02_02.txt#14', 'source_id': 'HtoWF_04_Part02_02.txt', 'text': 'as about to be graduated from Purdue University. After several phone conversations I learned that he had several offers from other companies, many of them larger and better known than mine. I was delighted when he accepted my offer. After he started on the job, I asked him why he had chosen us over ', 'score': 0.28579056, 'rrf_score': 0.0161290322580645

In [11]:
rrf_score

[{'chunk_id': 'HtoWF_04_Part02_02.txt#9',
  'source_id': 'HtoWF_04_Part02_02.txt',
  'text': 'ile that comes from within, the kind of smile that will bring a good price in the marketplace.\n\nProfessor James V. McConnell, a psychologist at the University of Michigan, expressed his feelings about a smile. "People who smile," he said, "tend to manage, teach and sell more effectively, and to rais',
  'score': 0.3120165,
  'rrf_score': 0.03278688524590164},
 {'chunk_id': 'HtoWF_04_Part02_02.txt#14',
  'source_id': 'HtoWF_04_Part02_02.txt',
  'text': 'as about to be graduated from Purdue University. After several phone conversations I learned that he had several offers from other companies, many of them larger and better known than mine. I was delighted when he accepted my offer. After he started on the job, I asked him why he had chosen us over ',
  'score': 0.28579056,
  'rrf_score': 0.016129032258064516},
 {'chunk_id': 'HtoWF_02WhythisBook.txt#32',
  'source_id': 'HtoWF_02WhythisBook.txt

In [12]:
tasks = [ask_rag(ques, rrf_score, k) for ques in qry]
responses = await asyncio.gather(*tasks)

In [13]:
responses

[{'question': 'Which university did psychologist Professor James V. McConnell belong to?',
  'answer': 'Professor James V. McConnell belonged to the University of Michigan [HtoWF_04_Part02_02.txt#9].',
  'sources': ['HtoWF_04_Part02_02.txt#9',
   'HtoWF_04_Part02_02.txt#14',
   'HtoWF_02WhythisBook.txt#32',
   'HtoWF_02WhythisBook.txt#14',
   'HtoWF_04_Part02_02.txt#30'],
  'tokens_in': 469,
  'tokens_out': 27,
  'retrieved': [{'chunk_id': 'HtoWF_04_Part02_02.txt#9',
    'source_id': 'HtoWF_04_Part02_02.txt',
    'text': 'ile that comes from within, the kind of smile that will bring a good price in the marketplace.\n\nProfessor James V. McConnell, a psychologist at the University of Michigan, expressed his feelings about a smile. "People who smile," he said, "tend to manage, teach and sell more effectively, and to rais',
    'score': 0.3120165,
    'rrf_score': 0.03278688524590164},
   {'chunk_id': 'HtoWF_04_Part02_02.txt#14',
    'source_id': 'HtoWF_04_Part02_02.txt',
    'text': 'as 

In [14]:
# Get Golden set questions
golden_folder = folder_path / "golden_set"

print(f"Checking directory: {golden_folder.resolve()}")
print(f"Does folder exist?: {golden_folder.exists()}\n")

if golden_folder.exists():
    print("Files found in this folder:")

golden = {
    row['GoldID']: row 
    for row in (json.loads(line) for line in (golden_folder / 'Golden_set_v1_3.jsonl').read_text().splitlines() if line.strip())}


Checking directory: /voc/work/IITM-AI-RAG/rag/docs/Dale_Carnigale/golden_set
Does folder exist?: True

Files found in this folder:


In [15]:
golden

{'GID0039': {'DocID': 'HtoWF_04_Part01_01.txt',
  'GoldID': 'GID0039',
  'Sno': 1,
  'Question': 'What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?',
  'Answer': '"I will speak ill of no man... and speak all the good I know of everybody."',
  'Citation': 'Doc-HtwWf_04_Part01_01|Para: Paragraph 30',
  'Answer_Type': 'Easy',
  'chunk_id': 'HtoWF_04_Part01_01.txt#44'},
 'GID0089': {'DocID': 'HtoWF_04_Part01_02.txt',
  'GoldID': 'GID0089',
  'Sno': 2,
  'Question': 'Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient?',
  'Answer': " Flattery is counterfeit praise designed to manipulate the listener for the flatterer's selfish ends; accepting it creates false pride and leaves one vulnerable to deception by insincere allies.",
  'Citation': 'Doc-HtwWf_04_Part01_02|Para: Paragraphs 42 & 44',
  'Answer_Type': 'Hard',
  'chunk_id': 'HtoWF_04_Part01_02.txt#37'},
 'GID0136': {'

In [16]:
#dense_tasks = []
#bm25_tasks = []
#responses = []

# 2. Build the task list (Do NOT use 'await' inside the loop)
#for item in golden.values():
#    query_text = item["Question"]
#
#    dense_results = await dense_search(query_text, coll_name, k)
#    bm25_results = await bm25_search(query_text, bm25_corpus, all_chunks, k)
#
#    rrf_score = rrf_fuse([dense_results, bm25_result], k=60, top_n=k)
#    responses.append(await ask_rag(query_text, rrf_score, k))


In [17]:
dense_tasks = []
bm25_tasks = []
responses = []

# Step 1: Collect all retrieval tasks without awaiting them yet
for item in golden.values():
    query_text = item["Question"]
    dense_tasks.append(dense_search(query_text, coll_name, k))
    bm25_tasks.append(bm25_search(query_text, bm25_corpus, all_chunks, k))

# Step 2: Concurrently execute all lookups at once
dense_responses = await asyncio.gather(*dense_tasks)
bm25_responses = await asyncio.gather(*bm25_tasks)

# Step 3: Loop through the results, fuse them, and call ask_rag
for item, dense_res, bm25_res in zip(golden.values(), dense_responses, bm25_responses):
    query_text = item["Question"]
    
    # Fuse the pre-fetched results
    rrf_score = rrf_fuse([dense_res, bm25_res], k=60, top_n=k)
    
    # Process RAG generation
    rag_response = await ask_rag(query_text, rrf_score, k)
    responses.append(rag_response)

In [18]:
responses[0]

{'question': 'What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?',
 'answer': 'The quote from Benjamin Franklin that summarizes his secret to handling people diplomatically is: "I will speak ill of no man, and speak all the good I know of everybody" [HtoWF_04_Part01_01.txt#44].',
 'sources': ['HtoWF_04_Part01_02.txt#0',
  'HtoWF_04_Part01_01.txt#44',
  'HtoWF_04_Part01_01.txt#43',
  'HtoWF_04_Part01_03.txt#2',
  'HtoWF_04_Part02_01.txt#18'],
 'tokens_in': 479,
 'tokens_out': 50,
 'retrieved': [{'chunk_id': 'HtoWF_04_Part01_02.txt#0',
   'source_id': 'HtoWF_04_Part01_02.txt',
   'text': '# Part One: Fundamental Techniques in Handling People\n\n## Chapter 2: The Big Secret of Dealing With People\n\nThere is only one way under high heaven to get anybody to do anything. Did you ever stop to think of that? Yes, just one way. And that is by **making the other person want to do it.**\n\nRemembe',
   'score': 0.44776565,
   'rrf_score': 0.032258064516129

In [19]:
for item, resp in zip(golden.values(), responses):
    hit_sources = resp.get("sources", [])
    resp["hit_rate"] = 1 if item["chunk_id"] in hit_sources else 0
    resp["cost"] = cost_usd(resp["tokens_in"], resp["tokens_out"])
    resp["chunk_id"] = item["chunk_id"]
    #resp["query"] = item["Question"]
    print(f"Item: {item['Question']} \n resp: {resp['question']} \n hit_sources:{hit_sources}\nItem_source:{item['chunk_id']}\nHit={resp['hit_rate']}")
    #result.append(resp)


Item: What quote from Benjamin Franklin summarizes his secret to handling people diplomatically? 
 resp: What quote from Benjamin Franklin summarizes his secret to handling people diplomatically? 
 hit_sources:['HtoWF_04_Part01_02.txt#0', 'HtoWF_04_Part01_01.txt#44', 'HtoWF_04_Part01_01.txt#43', 'HtoWF_04_Part01_03.txt#2', 'HtoWF_04_Part02_01.txt#18']
Item_source:HtoWF_04_Part01_01.txt#44
Hit=1
Item: Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient? 
 resp: Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient? 
 hit_sources:['HtoWF_04_Part01_02.txt#37', 'HtoWF_04_Part01_02.txt#36', 'HtoWF_04_Part02_01.txt#0', 'HtoWF_04_Part01_02.txt#38', 'HtoWF_04_Part01_01.txt#34']
Item_source:HtoWF_04_Part01_02.txt#37
Hit=1
Item: Why did White House head usher Ike Hoover say Rooseveltâ€™s visit was "the only happy day we had in nearly two year

In [20]:
for result in responses:
    print(f"\n Q: {result['question']}")
    #print(f"A: {result['answer']}")
    print(f"Hit rate: {result['hit_rate']}\nGolden Chunk_id: {result['chunk_id']} <==> \nSources retrieved: {result['sources']}")
    print(f"Prompt tokens: {result['tokens_in']} | Completion tokens: {result['tokens_out']} || Latency: {result['latency_s']}; Cost in USD:{result['cost']} ")
    


 Q: What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?
Hit rate: 1
Golden Chunk_id: HtoWF_04_Part01_01.txt#44 <==> 
Sources retrieved: ['HtoWF_04_Part01_02.txt#0', 'HtoWF_04_Part01_01.txt#44', 'HtoWF_04_Part01_01.txt#43', 'HtoWF_04_Part01_03.txt#2', 'HtoWF_04_Part02_01.txt#18']
Prompt tokens: 479 | Completion tokens: 50 || Latency: 1.1427486020002107; Cost in USD:6.0400000000000004e-05 

 Q: Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient?
Hit rate: 1
Golden Chunk_id: HtoWF_04_Part01_02.txt#37 <==> 
Sources retrieved: ['HtoWF_04_Part01_02.txt#37', 'HtoWF_04_Part01_02.txt#36', 'HtoWF_04_Part02_01.txt#0', 'HtoWF_04_Part01_02.txt#38', 'HtoWF_04_Part01_01.txt#34']
Prompt tokens: 518 | Completion tokens: 219 || Latency: 2.544935590998648; Cost in USD:0.00010655000000000001 

 Q: Why did White House head usher Ike Hoover say Rooseveltâ€™s visit was "the only happy day w

In [21]:
rate_summary = calc_golden_hit_cost_lat(responses)
print(rate_summary)

print(f"Hit rate: {(rate_summary['hit_rate']/rate_summary['total_hits'])*100} ( {rate_summary['hit_rate']}/{rate_summary['total_hits']}) | latency: {rate_summary['latency']} | Total Cost: ${rate_summary['total_cost_usd']}")


{'total_hits': 24, 'hit_rate': 22, 'latency': 33.47502829599762, 'total_cost_usd': 0.0015416000000000002}
Hit rate: 91.66666666666666 ( 22/24) | latency: 33.47502829599762 | Total Cost: $0.0015416000000000002


In [22]:

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

_reranker = None
def load_reranker():
    """Lazy-load the cross-encoder. Downloads ~80MB on first use."""
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        _reranker = CrossEncoder(RERANKER_MODEL)
    return _reranker


async def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """Score each (query, candidate.text) pair, return top_k by score.
    
    Candidates must have a 'doc' key with 'text' inside.
    Adds a 'rerank_score' field to each returned dict.
    """
    reranker = load_reranker()
    pairs = [(query, hit["text"]) for hit in candidates]
    print(f"Pairs: {pairs}")
    scores = reranker.predict(pairs)
    scored = [{**hit, "rerank_score": float(s)} for hit, s in zip(candidates, scores)]
    scored.sort(key=lambda h: h["rerank_score"], reverse=True)
    return scored[:top_k]


In [23]:
rerank_tasks = []
for resp in responses:
    retrieved_items = resp.get('retrieved', [])
    task = rerank(resp['question'], retrieved_items, 3)
    rerank_tasks.append(task)

# Fire all reranking queries to the model at the exact same time
rerank_list = await asyncio.gather(*rerank_tasks)

# Print your scores
for score in rerank_list:
    print(score)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Pairs: [('What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?', '# Part One: Fundamental Techniques in Handling People\n\n## Chapter 2: The Big Secret of Dealing With People\n\nThere is only one way under high heaven to get anybody to do anything. Did you ever stop to think of that? Yes, just one way. And that is by **making the other person want to do it.**\n\nRemembe'), ('What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?', 'France by living by this secret:\n\n"I will speak ill of no man, and speak all the good I know of everybody."\n\n"Any fool can criticize, condemn and complain—and most fools do. But it takes character and self-control to be understanding and forgiving."\n\n— Thomas Carlyle\n\nThe Lesson of Test Pilot Bob Ho'), ('What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?', 'tures bristling with prejudices and motivated by pride and vanity.\n\nBitter criti

In [24]:
rerank_score = [{**golden_item, "retreived": rerank_item} for golden_item, rerank_item in zip(golden.values(), rerank_list)]

In [25]:
rerank_score

[{'DocID': 'HtoWF_04_Part01_01.txt',
  'GoldID': 'GID0039',
  'Sno': 1,
  'Question': 'What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?',
  'Answer': '"I will speak ill of no man... and speak all the good I know of everybody."',
  'Citation': 'Doc-HtwWf_04_Part01_01|Para: Paragraph 30',
  'Answer_Type': 'Easy',
  'chunk_id': 'HtoWF_04_Part01_01.txt#44',
  'retreived': [{'chunk_id': 'HtoWF_04_Part01_02.txt#0',
    'source_id': 'HtoWF_04_Part01_02.txt',
    'text': '# Part One: Fundamental Techniques in Handling People\n\n## Chapter 2: The Big Secret of Dealing With People\n\nThere is only one way under high heaven to get anybody to do anything. Did you ever stop to think of that? Yes, just one way. And that is by **making the other person want to do it.**\n\nRemembe',
    'score': 0.44776565,
    'rrf_score': 0.03225806451612903,
    'rerank_score': -0.5365110039710999},
   {'chunk_id': 'HtoWF_04_Part01_01.txt#43',
    'source_id': 'HtoWF_04_Par

In [26]:
_rag_settings.GENERATE_ANSWER_RAG = True

In [34]:
rag_tasks = []

# 1. Loop through your list of 20 evaluation dictionaries
for evaluated_item in rerank_score:  # Replace with your actual list name
    query_text = evaluated_item['Question']
    
    # 2. Extract the nested list of chunks (which is keyed under 'rerank_score')
    chunks_to_sort = evaluated_item.get('retreived', [])
    
    # 3. Sort the 3 chunks by their inner 'rerank_score' descending
    sorted_reranked_chunks = sorted(
        chunks_to_sort, 
        key=lambda x: x.get('rerank_score', -999), 
        reverse=True
    )
    
    # 4. Build the concurrent RAG generation task
    task = ask_rag(query_text, sorted_reranked_chunks, k=3)
    rag_tasks.append(task)

# 5. Fire all 20 RAG generation requests to the LLM concurrently
final_rag_responses = await asyncio.gather(*rag_tasks)

# Print the final output generation strings
for idx, response in enumerate(final_rag_responses, start=1):
    print(f"--- Response {idx} ---")
    print(response)

--- Response 1 ---
{'question': 'What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?', 'answer': 'The quote from Benjamin Franklin that summarizes his secret to handling people diplomatically is: "I will speak ill of no man, and speak all the good I know of everybody" [HtoWF_04_Part01_01.txt#44].', 'sources': ['HtoWF_04_Part01_02.txt#0', 'HtoWF_04_Part01_01.txt#43', 'HtoWF_04_Part01_01.txt#44'], 'tokens_in': 314, 'tokens_out': 50, 'retrieved': [{'chunk_id': 'HtoWF_04_Part01_02.txt#0', 'source_id': 'HtoWF_04_Part01_02.txt', 'text': '# Part One: Fundamental Techniques in Handling People\n\n## Chapter 2: The Big Secret of Dealing With People\n\nThere is only one way under high heaven to get anybody to do anything. Did you ever stop to think of that? Yes, just one way. And that is by **making the other person want to do it.**\n\nRemembe', 'score': 0.44776565, 'rrf_score': 0.03225806451612903, 'rerank_score': -0.5365110039710999}, {'chunk_id': 'HtoWF_0

In [36]:
final_rag_responses

[{'question': 'What quote from Benjamin Franklin summarizes his secret to handling people diplomatically?',
  'answer': 'The quote from Benjamin Franklin that summarizes his secret to handling people diplomatically is: "I will speak ill of no man, and speak all the good I know of everybody" [HtoWF_04_Part01_01.txt#44].',
  'sources': ['HtoWF_04_Part01_02.txt#0',
   'HtoWF_04_Part01_01.txt#43',
   'HtoWF_04_Part01_01.txt#44'],
  'tokens_in': 314,
  'tokens_out': 50,
  'retrieved': [{'chunk_id': 'HtoWF_04_Part01_02.txt#0',
    'source_id': 'HtoWF_04_Part01_02.txt',
    'text': '# Part One: Fundamental Techniques in Handling People\n\n## Chapter 2: The Big Secret of Dealing With People\n\nThere is only one way under high heaven to get anybody to do anything. Did you ever stop to think of that? Yes, just one way. And that is by **making the other person want to do it.**\n\nRemembe',
    'score': 0.44776565,
    'rrf_score': 0.03225806451612903,
    'rerank_score': -0.5365110039710999},
   

In [38]:
for item, resp in zip(golden.values(), final_rag_responses):
    hit_sources = resp.get("sources", [])
    resp["hit_rate"] = 1 if item["chunk_id"] in hit_sources else 0
    resp["cost"] = cost_usd(resp["tokens_in"], resp["tokens_out"])
    resp["chunk_id"] = item["chunk_id"]
    #resp["query"] = item["Question"]
    print(f"Item: {item['Question']} \n resp: {resp['question']} \n hit_sources:{hit_sources}\nItem_source:{item['chunk_id']}\nHit={resp['hit_rate']}")
    #result.append(resp)

Item: What quote from Benjamin Franklin summarizes his secret to handling people diplomatically? 
 resp: What quote from Benjamin Franklin summarizes his secret to handling people diplomatically? 
 hit_sources:['HtoWF_04_Part01_02.txt#0', 'HtoWF_04_Part01_01.txt#43', 'HtoWF_04_Part01_01.txt#44']
Item_source:HtoWF_04_Part01_01.txt#44
Hit=1
Item: Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient? 
 resp: Analyze General Alvaro Obregonâ€™s warning: "Be afraid of the friends who flatter you." Why is flattery dangerous to the recipient? 
 hit_sources:['HtoWF_04_Part01_02.txt#37', 'HtoWF_04_Part01_02.txt#38', 'HtoWF_04_Part01_02.txt#36']
Item_source:HtoWF_04_Part01_02.txt#37
Hit=1
Item: Why did White House head usher Ike Hoover say Rooseveltâ€™s visit was "the only happy day we had in nearly two years"? 
 resp: Why did White House head usher Ike Hoover say Rooseveltâ€™s visit was "the only happy day we had in ne

In [39]:
rate_summary = calc_golden_hit_cost_lat(final_rag_responses)
print(rate_summary)

print(f"Hit rate: {(rate_summary['hit_rate']/rate_summary['total_hits'])*100} ( {rate_summary['hit_rate']}/{rate_summary['total_hits']}) | latency: {rate_summary['latency']} | Total Cost: ${rate_summary['total_cost_usd']}")


{'total_hits': 24, 'hit_rate': 22, 'latency': 31.223757697996916, 'total_cost_usd': 0.0010954499999999998}
Hit rate: 91.66666666666666 ( 22/24) | latency: 31.223757697996916 | Total Cost: $0.0010954499999999998


In [ ]:
#outpath = f"./response_output.txt"
#with open(outpath, "w", encoding="utf-8") as file:
#    for item in responses:
#        file.write(f"{item}\n")

In [ ]:
total_queries = len(golden)
hits_count = 0
sum_reciprocal_rank = 0.0

# Main Loop: Zip everything together row-by-row
for item, dense_resp, bm25_resp in zip(golden.values(), dense_responses, bm25_responses):
    
    # 1. Extract the ground truth (expected) chunk or file ID
    # Adjust the key string ("Ground_Truth" or "Expected_Chunk") to match your golden dataset format
    expected_chunk = item.get("Chunk_ID") or item.get("chunk_id") or item.get("Ground_Truth")
    
    # 2. Fuse the dense and sparse responses for this specific question
    rrf_score = rrf_fuse([dense_resp, bm25_resp], k=60, top_n=3)
    
    # 3. Hit Evaluation & MRR Logic
    is_hit = False
    for rank, resp in enumerate(rrf_score, start=1):
        # Extract the retrieved chunk identifier
        retrieved_chunk = resp.get("chunk_id")
        
        # Check if the retrieved chunk matches our ground truth
        if retrieved_chunk == expected_chunk:
            if not is_hit:
                hits_count += 1
                is_hit = True
            
            # Reciprocal rank is 1 / position (e.g., 1/1 for rank 1, 1/2 for rank 2)
            sum_reciprocal_rank += 1.0 / rank
            break  # Stop checking lower ranks once the correct chunk is found

# 4. Calculate Final Performance Metrics
hit_rate = (hits_count / total_queries) * 100 if total_queries > 0 else 0
mrr = sum_reciprocal_rank / total_queries if total_queries > 0 else 0

print(f"--- Evaluation Summary (Top-3 Hybrid RRF) ---")
print(f"Total Queries Evaluated: {total_queries}")
print(f"Hit Rate @ 3:            {hit_rate:.2f}%, {hits_count}")
print(f"MRR @ 3:                 {mrr:.4f}")

In [ ]:
# Dense search
topk_vec = await retrive_from_collection(q_vec, coll_name, k=3)
dense_results = [
    {
        "chunk_id": pt.payload.get("chunk_id"),
        "source_id": pt.payload.get("source_id"),
        "text": pt.payload.get("text"),
        "score": pt.score,
    }
    for pt in topk_vec
]
